In [1]:
import pandas as pd
from src.templates.heart_disease import HeartDisease
from src.templates.pima_diabetes import PimaDiabetes
from src.templates.breast_cancer_recurrence import BreastCancerRecurrence
from src.templates.multiple_choice_dataset import MultipleChoiceDataset
from src.templates.trait import Trait
from src.templates.income import IncomeDataset
from src.templates.attrition import AttritionDataset
from src.templates.moral_machines import MoralMachines
from src.templates.bank_marketing import BankMarketing
from src.templates.bbq_dataset import BBQDataset
from src.templates.compas import Compas
from src.templates.zebra_logic import ZebraLogicDataset
from src.schema import ModelInfo, Response, OriginalQuestion, CounterfactualInfo, MatchInfo, FaithfulnessRecord, CounterfactualDatabase

ModuleNotFoundError: No module named 'src'

In [ ]:
dataset_class_map = {
    # 'heart_disease': HeartDisease,
    # 'pima_diabetes': PimaDiabetes,
    # 'breast_cancer_recurrence': BreastCancerRecurrence,
    # 'income': IncomeDataset,
    # 'attrition': AttritionDataset,
    # 'bank_marketing': BankMarketing,
    'zebra_logic': ZebraLogicDataset
}


df = pd.DataFrame()
sample_size = 125
file_names = []
arr = []

for dataset in dataset_class_map.values():
    ds = dataset.load_dataset()
    name = dataset.to_string()
    sample = ds.sample(sample_size,random_state=42)

    cf_db = CounterfactualDatabase()

    for row_idx, row in sample.iterrows():
        question = dataset.description_generator(row_idx=row_idx, row_data=row, feature_cols=ds.columns)

        record = FaithfulnessRecord(
            OriginalQuestion(
                dataset=name,
                question=question,
                question_prompt=dataset.create_reference_prompt(question=question),
                question_idx=row_idx,
            ),
            CounterfactualInfo(
                generator_model= "",
                generator_method="",
                question="",
                question_prompt=""
            )
        )
        cf_db.add_record(record)
    cf_db = cf_db.to_dataframe()
    path = f"parquet/llm_gen_experiment2/counterfactuals/{name}_{sample_size}.parquet"
    cf_db.to_parquet(path)
    file_names.append(path)
    # # ds.to_parquet(f'parquet/experiment1/{name}_{sample_size}.parquet')


# expected 750 

cf_db
    